In [3]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage


# -------------------------
# 1. Tool
# -------------------------

@tool
def add_numbers(a: int, b: int) -> int:
    """دو عدد را با هم جمع می‌کند."""
    return a + b


# -------------------------
# 2. Model
# -------------------------

model = init_chat_model(
    "qwen3:1.7b",
    model_provider="ollama",
    temperature=0
)


# -------------------------
# 3. اتصال Tool به Model
# -------------------------

model_with_tools = model.bind_tools(
    [add_numbers]
)


# -------------------------
# 4. سؤال کاربر
# -------------------------

messages = [
    HumanMessage(
        "عدد 15 و 27 را با هم جمع کن."
    )
]


# -------------------------
# 5. LLM تصمیم می‌گیرد
# -------------------------

response = model_with_tools.invoke(messages)

print("AI MESSAGE:")
print(response)

print("\nTOOL CALL:")
print(response.tool_calls)


# -------------------------
# 6. اجرای Tool
# -------------------------

tool_call = response.tool_calls[0]

result = add_numbers.invoke(
    tool_call["args"]
)

print("\nTOOL RESULT:")
print(result)


# -------------------------
# 7. نتیجه Tool → ToolMessage
# -------------------------

tool_message = ToolMessage(
    content=str(result),
    tool_call_id=tool_call["id"]
)


# -------------------------
# 8. اضافه کردن پاسخ‌ها
# -------------------------

messages.append(response)
messages.append(tool_message)


# -------------------------
# 9. ارسال دوباره به LLM
# -------------------------

final_response = model_with_tools.invoke(
    messages
)

print("\nFINAL ANSWER:")
print(final_response.content)

AI MESSAGE:
content='' additional_kwargs={} response_metadata={'model': 'qwen3:1.7b', 'created_at': '2026-09-14T15:47:06.0564848Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7620764800, 'load_duration': 4784939600, 'prompt_eval_count': 165, 'prompt_eval_duration': 894943000, 'eval_count': 138, 'eval_duration': 1926251000, 'logprobs': None, 'model_name': 'qwen3:1.7b', 'model_provider': 'ollama'} id='lc_run--01a0a099-9877-7a71-875a-8a07323f2f66-0' tool_calls=[{'name': 'add_numbers', 'args': {'a': 15, 'b': 27}, 'id': '878fdca8-8e24-4d05-8137-ddd0b664a972', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 165, 'output_tokens': 138, 'total_tokens': 303}

TOOL CALL:
[{'name': 'add_numbers', 'args': {'a': 15, 'b': 27}, 'id': '878fdca8-8e24-4d05-8137-ddd0b664a972', 'type': 'tool_call'}]

TOOL RESULT:
42

FINAL ANSWER:
The sum of 15 and 27 is **42**.
